# TCC — Avaliação de Chunking p/ RAG (100% local)

Fluxo: **QA curado** (15 pares prontos, sem LLM) → **Fase 1** (LLaMA: fixed/recursive/semantic + baseline) → ⏸ reinicie a sessão → **Fase 2** (RAGAS, juiz Qwen-14B) → gráficos.
Cada fase roda em processo próprio e salva em disco (`analises/`). Se cair, reexecute só a fase que caiu.

In [ ]:
# 0) Confere GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) Entra no repo (funciona clonando ou abrindo o notebook pelo GitHub)
import os
if os.path.exists("src/config.py"):
    print("repo:", os.getcwd())
elif os.path.exists("chunking-evaluation/src/config.py"):
    %cd chunking-evaluation
else:
    !git clone https://github.com/murillodmf/chunking-evaluation.git
    %cd chunking-evaluation

In [ ]:
# 2) Anti-fragmentação de VRAM (rodar ANTES de qualquer uso da GPU)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("OK")

In [ ]:
# 3) Dependências
!pip install -q -r requirements.txt

In [ ]:
# 4) Login HF sem hardcodar token (use Colab Secrets ou cole aqui — não fica salvo no .ipynb commitado)
from getpass import getpass
from huggingface_hub import login
login(getpass("HF token: "))

In [ ]:
# 5) QA curado (sem LLM): confere os 15 pares prontos do gabarito
!python -c "import json; qa=json.load(open('Textos_exemplo/qa_milho_curado.json',encoding='utf-8')); print(f'{len(qa)} pares curados'); assert all(set(x)=={'question','ground_truth'} for x in qa); print('schema OK')"
# (Fase 0 gerativa/Qwen desativada p/ o milho: gabarito e curadoria, nao geracao)

In [ ]:
# 6) Fase 1 — geração (GPU, LLaMA 3.1 8B): 3 chunkers + baseline, 15 perguntas
!python src/02_gerar_respostas.py --doc milho --amostra todas
# smoke: trocar por --amostra 3 | sem baseline: acrescentar --sem-baseline

## ⏸ Pare aqui: **Ambiente de execução → Reiniciar sessão**

Depois reexecute as células 1–4 (rápidas) e siga na Fase 2 com a VRAM limpa. Os JSONs da Fase 1 já estão salvos em `analises/`.

In [ ]:
# 7) Fase 2 — notas RAGAS (GPU, Qwen-14B juiz), sessão nova
!python src/03_avaliar_ragas.py --doc milho
!python src/03_avaliar_ragas.py --doc milho --strategy baseline --metrics answer_relevancy,answer_correctness

In [ ]:
# 8) Resultados
import pandas as pd
summary = pd.read_csv("analises/ragas_summary_milho.csv", index_col=0)
display(summary)
ax = summary.plot(kind="bar", figsize=(9, 4), rot=0)
ax.set_ylim(0, 1); ax.set_title("RAGAS por estratégia — milho"); ax.grid(axis="y", alpha=0.3)